# ReCAP study — first run (Google Colab)

Use this notebook to verify the upstream ReCAP stack on Colab before heavier NAS100 pilots.

**What it does:** clone upstream (or your fork), install deps, run a **tiny synthetic** ReCAP pass.

After your study code is on GitHub, change `REPO_URL` to your fork.

In [ ]:
# Config
REPO_URL = "https://github.com/Dumail/ReCAP.git"
REPO_DIR = "/content/ReCAP"
BRANCH = "main"

In [ ]:
import os, sys, subprocess
from pathlib import Path

if not Path(REPO_DIR).exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())
subprocess.check_call(["git", "rev-parse", "--short", "HEAD"])

In [ ]:
%pip install -q -r requirements.txt pytest

In [ ]:
# Prefer study smoke if present; else inline synthetic run
from pathlib import Path
smoke = Path("study/smoke_run.py")
if smoke.exists():
    %run study/smoke_run.py
else:
    print("study/smoke_run.py not found — using upstream-style tiny run")
    import pandas as pd
    from recap.data import add_market_features
    from recap.recap import ReCAP, ReCAPConfig

    dates = pd.date_range("2020-01-01", periods=60, freq="B")
    rows = []
    for tic_i, tic in enumerate(["AAA", "BBB", "CCC"]):
        base = 100 + 5 * tic_i
        for d_i, d in enumerate(dates):
            close = base * (1.002 if d_i < 30 else 0.999) ** d_i
            rows.append({"date": d, "tic": tic, "open": close*0.99, "high": close*1.01,
                         "low": close*0.98, "close": close, "volume": 1e6})
    data = add_market_features(pd.DataFrame(rows), use_vix=False)
    train = data[data["date"] < "2020-03-01"]
    test = data[data["date"] >= "2020-03-01"]
    runner = ReCAP(
        config=ReCAPConfig(initial_cash=1000, pretrain_timesteps=16, finetune_timesteps=8,
                           n_steps=8, batch_size=8, min_segment_days=8, gate_epochs=5, device="cpu"),
        tickers=["AAA", "BBB", "CCC"], use_vix=False,
    )
    print(runner.run(train, test, online_update=False)["metrics"])

## Next steps

1. Point `REPO_URL` at your fork once study code is pushed.
2. For a reduced DOW30/NAS100 run, call `main.py` with small `--pretrain-timesteps` / `--finetune-timesteps`.
3. Keep full paper budgets and multi-seed sweeps for Colab Pro / a stronger machine.